# Financial Health Scoring

### Objective
The goal of this phase is to design an **explainable and scalable financial health scoring framework** that combines business rules with normalized behavioral metrics.

The resulting score:
* Ranges from 0 to 100
* Allows ranking of accounts by financial stability
* Supports categorization into Healthy, Moderate, and At-Risk segments
* Is interpretable by non-technical stakeholders

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load aggregated dataset from notebook_2
df_health = pd.read_csv("../outputs/account_month_metrics.csv")

### Feature Selection
Financial health scoring is built using **interpretable, behavior-driven fetaures** derived from account-level monthly activity.

The selected featuress reflect five core dimensions:
* Liquidity strength
* Cash flow sustainability
* Financial stability
* Engagement consistency
* Liquidity risk

In [4]:
# Ensuring required columns exist
required_cols = [
    "Account No",
    "year_month",
    "txn_count",
    "total_inflow",
    "total_outflow",
    "balance_std",
    "unique_merchants"
]

missing = set(required_cols) - set(df_health.columns)
missing

set()

#### Net Cash Flow

In [6]:
df_health["net_cash_flow"] = (
    df_health["total_inflow"] + df_health["total_outflow"]
)

#### Average Monthly Balance

In [8]:
# Proxy for average balance
df_health["avg_balance"] = (
    df_health["total_inflow"] + df_health["total_outflow"]
).cumsum()

**Note** In the absence of true monthly average balance field, a proxy is derived to maintain interpretability while avoiding leakage.

#### Low-Balance Risk Flag

In [11]:
LOW_BALANCE_THRESHOLD = 500

df_health["low_balance_flag"] = (
    df_health["avg_balance"] < LOW_BALANCE_THRESHOLD
).astype(int)

### Normalization Strategy
To combine heterogeneous financial metrics into a single score, all features are normalized to a 0-1 range using Min-Max scaling.

Metric directionality is handled explicitly so that:
* Higher normalized values always indicate better financial health

In [13]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df_health["balance_score"] = scaler.fit_transform(
    df_health[["avg_balance"]]
)

df_health["cashflow_score"] = scaler.fit_transform(
    df_health[["net_cash_flow"]]
)

df_health["volatility_score"] = 1 - scaler.fit_transform(
    df_health[["balance_std"]].fillna(0)
)

df_health["txn_score"] = scaler.fit_transform(
    df_health[["txn_count"]]
)

df_health["low_balance_score"] = 1 - scaler.fit_transform(
    df_health[["low_balance_flag"]]
)

### Weighted Financial Health Score
Normalized financial indicators are combined using business-defined weights. Weights reflect importance rather than statistical optimization, ensuring transparency and interpretability.

#### Score Threshold Definition
Health category thresholds were defined to reflect how a product or finance team might operationalize financial health monitoring in the absence of labeled outcomes.

Thresholds were informed by:
* Observed behavioral distributions
* Conservative risk sensitivity
* Industry-standard financial health practices

These thresholds are intended as a starting point and would typically be refined over time using outcome-based validation and stakeholder input.

In [15]:
df_health["financial_health_score"] = (
    0.30 * df_health["balance_score"] +
    0.25 * df_health["cashflow_score"] +
    0.20 * df_health["volatility_score"] +
    0.15 * df_health["txn_score"] +
    0.10 * df_health["low_balance_score"]
) * 100

### Rule-Based Health Categories
To maintain interpretability, continuous financial health scores are mapped to discrete health categories using business-defined thresholds.

In [17]:
def assign_health_category(score):
    if score >= 70:
        return "Healthy"
    elif score >= 40:
        return "Moderate"
    else:
        return "At-Risk"

df_health["health_category"] = df_health["financial_health_score"].apply(
    assign_health_category
)

### Validation & Sanity Checks
Validation checks are performed to ensure:
* Financial health scores fall within expected bounds
* Category distributions are reasonable
* Results align with behavioral insights observed in EDA.

In [19]:
df_health["financial_health_score"].describe()

count    13555.000000
mean        40.693677
std          8.382531
min         13.381988
25%         35.250282
50%         39.335330
75%         47.528109
max         77.912167
Name: financial_health_score, dtype: float64

The financial health scores range from approximately **13 to 78**, with a mean score of **~41**.

#### Key Observations
* The score distribution is **left-skewed**, with most accounts clustering in the lower score ranges
* The interquartile range (25th-75th percentile) lies between **35 and 48**, indicating limited variance for the majority of users
* Very few accounts achieve scores above 70, suggesting strong financial health is uncommon in the dataset

#### Interpretation
The concentration of scores in the lower range reflects behavioral patterns observed in EDA:
* Persistent net outflows exceeding inflows
* High balance volatility
* Declining merchant diversity over time

These characteristics collectively depress financial health scores and are consistent with a population experiencing liquidity pressure rather than long-term balance growth.

In [21]:
df_health["health_category"].value_counts(normalize=True)

health_category
At-Risk     0.540317
Moderate    0.459240
Healthy     0.000443
Name: proportion, dtype: float64

Accounts are classified into health categories using business-defined score thresholds.
* **At-Risk:** ~54%
* **Moderate:** ~46%
* **Healthy:** <1%

#### Business Interpretation
The dominance of At-Risk and Moderate categories indicates that most users operate with:
* Thin liquidity buffers
* Limited positive cash flow
* High month-to-month instability

The near absence of Healthy accounts is not necessarily a modeling flaw, but rather a reflection of conservative scoring thresholds and spending-heavy user behavior.

In [23]:
df_health["Account No"] = df_health["Account No"].astype(int)

# Saving final health scoring dataset
df_health.to_csv(
    "../outputs/account_financial_health.csv",
    index=False
)

print("Financial health dataset saved to outputs/account_financial_health.csv")

Financial health dataset saved to outputs/account_financial_health.csv


### Account-Level Financial Health Validation
#### Objective
To demonstrate that the financial health score is:
* Intuitively correct
* Behaviorally grounded
* Interpretable at an individual account level

#### Select Representative Accounts
The goal is to confirm that score assignments align with observed transactional and balance behaviors at an individual account level.

In [26]:
# Selecting representative accounts from each category
example_accounts = (
    df_health
    .sort_values("financial_health_score")
    .groupby("health_category")
    .head(1)
    [["Account No", "health_category", "financial_health_score"]]
)

example_accounts

,Account No,health_category,financial_health_score
12902,952609703,At-Risk,13.381988
8098,633094830,Moderate,40.000618
752,147241590,Healthy,71.006318


* Clear **score separation** across categories
* Healthy score is *meaningfully higher*, not just marginally
* At-Risk score is near the **lower bound**, which is expected

This indicates the score distribution is behaving sensibly.

#### Extract Score History for Selected Accounts
For each selected account, the financial health score is examined over time to assess behavioral consistency and temporal stability.

In [28]:
selected_account_ids = example_accounts["Account No"].unique()

account_timelines = (
    df_health[df_health["Account No"].isin(selected_account_ids)]
    .sort_values(["Account No", "year_month"])
)

account_timelines.head()

,Account No,year_month,txn_count,total_inflow,total_outflow,avg_txn_amount,balance_std,unique_merchants,net_cash_flow,avg_balance,low_balance_flag,balance_score,cashflow_score,volatility_score,txn_score,low_balance_score,financial_health_score,health_category
752,147241590,2025-01,35,45000.0,-10987.49,971.78600,1749.500460,9,34012.51,-172100.90,1,0.891840,0.903945,0.684187,0.531250,0.0,71.006318,Healthy
753,147241590,2025-02,16,0.0,-1702.46,-106.40375,1513.087426,7,-1702.46,-173803.36,1,0.890958,0.286958,0.726864,0.234375,0.0,51.955592,Moderate
754,147241590,2025-03,3,0.0,-170.34,-56.78000,1054.206437,3,-170.34,-173973.70,1,0.890870,0.313426,0.809699,0.031250,0.0,51.224475,Moderate
755,147241590,2025-04,5,0.0,-230.83,-46.16600,2160.622476,4,-230.83,-174204.53,1,0.890751,0.312381,0.609973,0.062500,0.0,47.669001,Moderate
756,147241590,2025-05,2,0.0,-115.72,-57.86000,465.608232,2,-115.72,-174320.25,1,0.890691,0.314369,0.915950,0.015625,0.0,53.133335,Moderate


**Account No 147241590**
* **Jan (Healthy)** month shows **strong income dominance**, active usage and diversified merchant behavior.
    * Net Cash flow: +34,012
    * Transaction Volume: 35
    * Merchants: 9
    * Balance volatility: moderate
    * Score: 71.0
* **Feb** $\rightarrow$ **April (Moderate)**
    * Net Cash flow turns negative
    * Transaction count drops sharply
    * Scores fall to ~48-52
    * Balance remains large but direction deteriorates

#### Behavioral Comparison by Health Category
This section aggregates monthly financial health signals at the **account level** to understand long-term behavioral patterns behind each health category. The goal is to validate whether the hybrid scoring framework produces **consistent**, **interpretable account profiles** over time.

**Aggregated Metrics Used**
* `avg_score` - Mean financial health score across all active months
* `avg_balance` - Average account balance (indicator of financial buffer)
* `avg_cashflow` - Mean net cash flow (inflow - outflow)
* `avg_volatility` - Average balance standard deviation (financial stability)
* `avg_txn_count` - Average monthly transaction frequency
* `low_balance_rate` - Proportion of months flagged as low balance

In [31]:
account_summary = (
    account_timelines
    .groupby(["Account No", "health_category"])
    .agg(
        avg_score=("financial_health_score", "mean"),
        avg_balance=("avg_balance", "mean"),
        avg_cashflow=("net_cash_flow", "mean"),
        avg_volatility=("balance_std", "mean"),
        avg_txn_count=("txn_count", "mean"),
        low_balance_rate=("low_balance_flag", "mean")
    )
    .reset_index()
)

account_summary

,Account No,health_category,avg_score,avg_balance,avg_cashflow,avg_volatility,avg_txn_count,low_balance_rate
0,147241590,Healthy,71.006318,-1.721009e+05,34012.510000,1749.500460,35.000000,1.0
1,147241590,Moderate,51.707366,-1.748318e+05,-360.500000,1258.586361,5.090909,1.0
2,633094830,At-Risk,38.066140,-1.237926e+06,-716.079000,1349.112198,22.900000,1.0
3,633094830,Moderate,46.447318,-1.236095e+06,12776.240000,1435.057162,35.000000,1.0
4,952609703,At-Risk,18.775939,-1.822247e+06,-1119.348333,3233.680594,9.083333,1.0


#### Healthy Accounts
Demonstrate **strong income capacity and active engagement**, even if balances are not consistently positive.

Key Characteristics:
* **Highest average financial health score**
* **Strong positive net cash flow**, indicating inflow dominance
* **High transaction activity**, reflecting regular account usage
* Moderate volatility, consistent with higher transaction volume

The model correctly prioritizes **cash flow strength and engagement** over static balance levels, aligning with real-world financial health assessment.

#### Moderate Accounts
Represent **financially stable but constrained behavior**

Observed patterns:
* Mid-range financial health scores
* Near-neutral or slightly negative cash flow
* Reduced transaction frequency
* Moderate balance volatility

These accounts show **no acute distress**, but lack the inflow momentum required to qualify as Healthy. The model appropriately captures this gray zone rather than forcing binary classification.

#### At-Risk Accounts
Exhibit **persistent financial stress** across multiple dimensions.

Key Indicators:
* Lowest average financial health scores
* Sustained negative cash flow
* High balance volatility
* Limited recovery periods

The hybrid scoring framework correctly identifies **structural financial instability**, not just short-term anomalies.

#### `low_balance_flag = 1.0` across all accounts
**Why this occurs**:
* The low balance flag is currently defined using a **global absolute threshold**
* The dataset contains **heavily negative average balances** across most accounts
* As a result, the threshold is triggered consistently

**Why this does *not* invalidate the model**:
* The low balance component is **only one weighted factor**
* Other dimensions (cash flow, volatility, transaction behavior) still drive differentiation
* Health categories are not determined by this flag alone

**Future improvements could include**
* Replacing global thresholds with:
    * Account-relative thresholds (e.g., percentile-based)
    * Rolling historical baselines
 
### Overall Validation Outcome
The account-level summaries confirm that:
* Financial health scores reflect **consistent behavioral patterns**
* Category assignments are **interpretable and defensible**
* The hybrid scoring approach successfully balances rules with normalized signals

This validates the financial health scoring framework as a **robust, explainable foundation** suitable for business analytics and product decision-making.